# Sesión 5: Lenguaje DAX y funciones de cálculo## Ejemplos prácticos y funciones de Power BIEste notebook contiene ejemplos prácticos de DAX para copiar y pegar directamente en Power BI Desktop.

---## SLIDE 9: Creación de Columnas Calculadas### ConceptoLas columnas calculadas son campos adicionales que se crean dentro de una tabla y se calculan fila por fila utilizando DAX.**Características:**- Se almacenan físicamente en el modelo (consumen memoria)- Se evalúan una vez al cargar- Pueden usarse en filtros y relaciones- Ideales para clasificaciones y derivaciones### Ejemplo 1: Clasificación simple por duración

```daxClasificacion_Tiempo = IF(    Solicitudes[DuracionHoras] <= 48,    "Rápida",    "Lenta")```**Explicación:**- `IF()`: Si la condición es verdadera, retorna "Rápida", si no, "Lenta"- Se evalúa para cada fila de la tabla Solicitudes- Ahora puedes filtrar o agrupar por esta clasificación

### Ejemplo 2: Clasificación multinivel (anidada)

```daxPrioridad_Operativa = IF(    Solicitudes[DuracionHoras] < 24,    "Crítica",    IF(        Solicitudes[DuracionHoras] < 48,        "Alta",        IF(            Solicitudes[DuracionHoras] < 72,            "Normal",            "Baja"        )    ))```**Explicación:**- IF anidados crean una cascada de condiciones- Resultado: 4 categorías diferentes según tiempo

### Ejemplo 3: Concatenación de valores

```daxNombre_Completo = CONCATENATE(    Solicitudes[Nombre],    " ",    Solicitudes[Apellido])```**O más moderno (& es más rápido):**```daxNombre_Completo = Solicitudes[Nombre] & " " & Solicitudes[Apellido]```

### Ejemplo 4: Crear código único

```daxCodigo_Solicitud = Solicitudes[ID_Comuna] & "-" & TEXT(Solicitudes[FechaCreacion], "YYYYMM") & "-" &FORMAT(Solicitudes[ID_Solicitud], "0000")```**Resultado:** "1-202401-0001", "3-202402-0045"### Buenas prácticas✓ Usar para clasificaciones o derivaciones✓ Nombres descriptivos✓ Validar en todas las filas✗ NO usar cuando medida sería más eficiente✗ NO depender de contexto de filtro

---## SLIDE 12-16: Creación de Medidas Calculadas### ConceptoLas medidas son expresiones DAX que realizan cálculos agregados y se evalúan en tiempo real según el contexto del filtro.**Características:**- NO se almacenan por fila (muy eficientes)- Se recalculan dinámicamente según filtros- Ideales para agregaciones y KPIs- Responden automáticamente a cambios de visualización### Ejemplo 1: Contar registros

```daxTotal_Solicitudes = COUNTROWS(Solicitudes)```**Uso:**- En tarjeta: mostrará el total dinámicamente- Si aplicas filtro por Comuna, mostrará solo esa comuna- Si aplicas filtro por Mes, mostrará solo ese mes

### Ejemplo 2: Promedio

```daxPromedio_Duracion = AVERAGE(Solicitudes[DuracionHoras])```**Resultado:** 42.5 (promedio de horas)

### Ejemplo 3: Con condición específica

```daxSolicitudesCerradas = CALCULATE(    COUNTROWS(Solicitudes),    Solicitudes[Estado] = "Cerrada")```**Explicación:**- `CALCULATE()`: Cambia el contexto de filtro- Cuenta SOLO filas donde Estado = "Cerrada"- Independientemente del filtro aplicado en la visualización

### Ejemplo 4: Tasa de resolución

```daxTasa_Resolucion = DIVIDE(    [SolicitudesCerradas],    [Total_Solicitudes],    0  // Valor si hay división por cero)```**Explicación:**- `DIVIDE()`: Divide dos medidas de forma segura- Si Total es 0, devuelve 0 en lugar de error- Resultado: 0.85 (85% de solicitudes cerradas)

### Ejemplo 5: Solicitudes rápidas

```daxSolicitudes_Rapidas = CALCULATE(    COUNTROWS(Solicitudes),    Solicitudes[DuracionHoras] < 24)```### Buenas prácticas✓ Nombres claros y descriptivos✓ Documentar medidas complejas con comentarios✓ Usar DIVIDE en lugar de /✓ Reutilizar medidas en otras medidas✓ Agrupar en carpetas por tema✗ NO evaluar fila por fila (usar columnas)✗ NO ignorar el contexto de filtro

---## SLIDE 18-22: Creación de Tablas Calculadas### ConceptoLas tablas calculadas son entidades nuevas creadas mediante DAX que contienen múltiples columnas y filas.**Características:**- Se comportan como tablas importadas- Pueden tener relaciones- Útiles para tablas puente, resúmenes, calendarios- Se actualizan al refrescar el modelo### Ejemplo 1: Filtrar solicitudes por estado

```daxSolicitudes_Cerradas = FILTER(    Solicitudes,    Solicitudes[Estado] = "Cerrada")```**Resultado:**- Nueva tabla con SOLO solicitudes cerradas- Mantiene todas las columnas originales- Puede usarse en visualizaciones o relaciones

### Ejemplo 2: Tabla de resumen por servicio

```daxResumen_Servicio = SUMMARIZE(    Solicitudes,    Solicitudes[TipoServicio],    "Total", COUNTROWS(Solicitudes),    "Promedio_Dias", AVERAGE(Solicitudes[DuracionDias]))```**Resultado:**| TipoServicio | Total | Promedio_Dias ||--------------|-------|---------------|| Reparación   | 45    | 2.3           || Alumbrado    | 32    | 1.8           || Recolección  | 28    | 1.2           |

### Ejemplo 3: Tabla calendario

```daxCalendario = CALENDAR(    DATE(2024, 1, 1),    DATE(2024, 12, 31))```**Luego agrega columnas:**```daxCalendario[Año] = YEAR(Calendario[Date])Calendario[Mes] = MONTH(Calendario[Date])Calendario[Mes_Nombre] = FORMAT(Calendario[Date], "MMMM")Calendario[Trimestre] = ROUNDUP(MONTH(Calendario[Date])/3, 0)```### Usos comunes- Tablas puente para relaciones N:N- Resúmenes de datos- Calendarios personalizados- Clasificaciones auxiliares

---## SLIDE 25-28: Funciones Clave Más Utilizadas### 1. COUNTROWS - Contar registros```daxTotal_Solicitudes = COUNTROWS(Solicitudes)```**Variantes:**```dax// Contar solo cerradasCerradas = COUNTROWS(FILTER(Solicitudes, Solicitudes[Estado] = "Cerrada"))// Contar únicosSolicitudes_Unicas = COUNTROWS(DISTINCT(Solicitudes, Solicitudes[ID_Ciudadano]))```

### 2. RELATED - Acceder a tabla relacionada```daxNombreComuna = RELATED(Comunas[Nombre])```**Uso:** En una columna calculada en Solicitudes- Lee desde la tabla Comunas (relacionada)- Requiere relación 1:N

### 3. CALCULATE - Modificar contexto de filtro```daxSolicitudesCerradas = CALCULATE(    COUNTROWS(Solicitudes),    Solicitudes[Estado] = "Cerrada")```**Múltiples condiciones:**```daxSolicitudes_Urgentes = CALCULATE(    COUNTROWS(Solicitudes),    Solicitudes[Prioridad] = "Alta",    Solicitudes[DuracionHoras] < 24)```

### 4. DIVIDE - Dividir de forma segura```daxTasa_Resolucion = DIVIDE(    [SolicitudesCerradas],    [TotalSolicitudes],    0  // Valor si hay división por cero)```**Comparación:**```dax// MAL - riesgo de errorTasa = [Cerradas] / [Total]// BIEN - seguroTasa = DIVIDE([Cerradas], [Total], 0)```

---## SLIDE 30-33: Funciones de Inteligencia de Tiempo### Requisito previo: Tabla CalendarioPrimero, crea una tabla calendario:```daxCalendario = CALENDAR(DATE(2020, 1, 1), DATE(2025, 12, 31))```Luego añade columnas:```daxCalendario[Año] = YEAR(Calendario[Date])Calendario[Mes] = MONTH(Calendario[Date])Calendario[Mes_Nombre] = FORMAT(Calendario[Date], "MMMM")```Finalmente, crea una relación: `Solicitudes[Fecha] → Calendario[Date]`### Ejemplo 1: TOTALYTD (Acumulado anual)

```daxTotal_YTD = TOTALYTD(    [Total_Solicitudes],    Calendario[Date])```**Resultado:** Acumulado desde enero hasta la fecha actual en cada mes

### Ejemplo 2: Comparación con año anterior```daxSolicitudes_AñoAnterior = CALCULATE(    [Total_Solicitudes],    SAMEPERIODLASTYEAR(Calendario[Date]))```**Resultado:** Mismo mes pero del año anterior

### Ejemplo 3: Variación año a año```daxVariacion_YoY = DIVIDE(    [Total_Solicitudes] - [Solicitudes_AñoAnterior],    [Solicitudes_AñoAnterior],    0)```**Resultado:** Porcentaje de cambio

### Ejemplo 4: Mes anterior```daxSolicitudes_MesAnterior = CALCULATE(    [Total_Solicitudes],    DATEADD(Calendario[Date], -1, MONTH))```### Ejemplo 5: Mes actual (MTD)```daxSolicitudes_MesActual = CALCULATE(    [Total_Solicitudes],    DATESMTD(Calendario[Date]))```### Funciones de tiempo disponibles- `TOTALYTD()`: Total año a la fecha- `SAMEPERIODLASTYEAR()`: Mismo periodo año anterior- `DATEADD()`: Desplazar fechas- `DATESMTD()`, `DATESQTD()`, `DATESYTD()`: Filtrar por mes, trimestre, año

---## SLIDE 35-37: Funciones Iterativas X### ConceptoLas funciones terminadas en X iteran sobre una tabla evaluando una expresión en cada fila.### Ejemplo 1: SUMX - Suma con cálculo por fila```daxCosto_Total = SUMX(    Solicitudes,    Solicitudes[Cantidad] * Solicitudes[PrecioUnitario])```**Explicación:**- Por cada fila en Solicitudes- Multiplica Cantidad × PrecioUnitario- Suma todos los resultados

### Ejemplo 2: AVERAGEX - Promedio iterativo```daxPromedio_Ponderado = AVERAGEX(    Solicitudes,    Solicitudes[Calificacion] * Solicitudes[Peso])```

### Ejemplo 3: MAXX y MINX - Máximo y mínimo```daxDuracion_Maxima = MAXX(    Solicitudes,    Solicitudes[DuracionHoras])Duracion_Minima = MINX(    Solicitudes,    Solicitudes[DuracionHoras])```### Ejemplo 4: COUNTX - Contar condiciones```daxServicios_Activos = COUNTX(    FILTER(Servicios, Servicios[Estado] = "Activo"),    Servicios[ID])```**Uso:** Cuando necesitas iteración con condiciones específicas

---## SLIDE 39-41: Funciones Condicionales### Ejemplo 1: IF```daxCategoria_Duracion = IF(    Solicitudes[DuracionHoras] > 48,    "Lenta",    "Rápida")```### Ejemplo 2: IF anidado```daxEs_Urgente = IF(    AND(        Solicitudes[DuracionHoras] < 24,        Solicitudes[Prioridad] = 1    ),    TRUE,    FALSE)```

### Ejemplo 3: SWITCH - Mejor que IF múltiple```dax// MALO - muchos IF anidadosCategoria = IF([Prioridad] = 1, "Alta",    IF([Prioridad] = 2, "Media",        IF([Prioridad] = 3, "Baja", "Sin definir")    ))// BIEN - SWITCH es más legibleCategoria = SWITCH(    Solicitudes[Prioridad],    1, "Alta",    2, "Media",    3, "Baja",    "Sin definir")```### Ejemplo 4: IFERROR```daxTasa_Resolucion = IFERROR(    DIVIDE([Cerradas], [Total]),    0)```### Ejemplo 5: Operadores lógicos```daxEs_Caso_Especial = IF(    (Solicitudes[Prioridad] = 1) && (Solicitudes[DuracionHoras] > 72),    TRUE,    FALSE)```**Operadores:**- `&&` u `AND()`: Ambas verdaderas- `||` u `OR()`: Al menos una verdadera- `NOT()`: Negación

---## SLIDE 43-45: Funciones de Texto### Ejemplo 1: CONCATENATE```daxNombre_Completo = CONCATENATE(    Solicitudes[Nombre],    " ",    Solicitudes[Apellido])```**O más eficiente:**```daxNombre_Completo = Solicitudes[Nombre] & " " & Solicitudes[Apellido]```

### Ejemplo 2: LEFT, RIGHT, MID```daxCodigo_Area = LEFT(Solicitudes[Telefono], 3)        // Primeros 3 dígitosUltimos_Digitos = RIGHT(Solicitudes[Telefono], 4)   // Últimos 4 dígitosSubcadena = MID(Solicitudes[Observaciones], 1, 20)  // Del 1 al 20```

### Ejemplo 3: UPPER, LOWER, FORMAT```daxNombre_Mayuscula = UPPER(Solicitudes[Nombre])Nombre_Minuscula = LOWER(Solicitudes[Nombre])Fecha_Formateada = FORMAT(Solicitudes[FechaCreacion], "DD-MM-YYYY")```

### Ejemplo 4: SUBSTITUTE - Reemplazar```daxTexto_Limpio = SUBSTITUTE(    Solicitudes[Observaciones],    "-",    " ")```Reemplaza todos los "-" por espacios

### Ejemplo 5: CONCATENATEX - Iterar y concatenar```daxServicios_Usuario = CONCATENATEX(    FILTER(        Solicitudes,        Solicitudes[IDUsuario] = 123    ),    Solicitudes[Servicio],    ", ")```**Resultado:** "Reparación, Alumbrado, Recolección" (concatenados con coma)

---## SLIDE 48-49: Funciones Estadísticas### Ejemplo 1: AVERAGE```daxPromedio_Duracion = AVERAGE(Solicitudes[DuracionHoras])```### Ejemplo 2: MEDIAN```daxMediana_Duracion = MEDIAN(Solicitudes[DuracionHoras])```**Diferencia:**- Promedio: suma/cantidad (sensible a extremos)- Mediana: valor central (más robusto)

### Ejemplo 3: MIN y MAX```daxDuracion_Minima = MIN(Solicitudes[DuracionHoras])Duracion_Maxima = MAX(Solicitudes[DuracionHoras])```### Ejemplo 4: Desviación estándar```dax// Desviación poblacionalDesviacion_Poblacion = STDEV.P(Solicitudes[DuracionHoras])// Desviación muestralDesviacion_Muestra = STDEV.S(Solicitudes[DuracionHoras])```**Uso:**- Poblacional (STDEV.P): Todos los datos- Muestral (STDEV.S): Parte de los datos### Ejemplo 5: Varianza```daxVarianza_Poblacion = VAR.P(Solicitudes[DuracionHoras])Varianza_Muestra = VAR.S(Solicitudes[DuracionHoras])```### Interpretación- Promedio: Centro de distribución- Desviación: Dispersión alrededor del promedio- Varianza: Cuadrado de la desviación

---## Buenas Prácticas Generales en DAX### Nomenclatura```dax// BIENTotal_Solicitudes_Mensuales = ...Tasa_Resolucion = ...Promedio_Duracion_Horas = ...// MALMedida1 = ...temp = ...X = ...```### Documentación```dax// Calcula la tasa de resolución de solicitudes// Filtrado por estado = "Cerrada"// Autor: Equipo BI | Fecha: 2024-01-15Tasa_Resolucion = DIVIDE(    [SolicitudesCerradas],    [TotalSolicitudes],    0)```### Usar VAR para claridad```dax// Difícil de leerResultado = DIVIDE([A], [A] + [B], 0) * 100// Claro y legibleResultado = VAR Total = [A] + [B]VAR Porcentaje = DIVIDE([A], Total, 0)RETURN Porcentaje * 100```### Preferencias✓ DIVIDE en lugar de /✓ SWITCH en lugar de IF múltiple✓ Medidas en lugar de columnas para agregaciones✓ Nombres descriptivos✓ Comentarios en expresiones complejas✓ CONCATENATE o & para texto✗ Nombres técnicos o abreviados✗ Lógica duplicada✗ Anidar demasiados IF✗ Ignorar el contexto de filtro✗ División sin manejo de errores

---## Resumen de Funciones por Categoría### Agregación- COUNTROWS, SUM, AVERAGE, MIN, MAX, MEDIAN### Relacionales- RELATED, RELATEDTABLE### Modificación de contexto- CALCULATE, CALCULATETABLE### Iterativas (terminan en X)- SUMX, AVERAGEX, MAXX, MINX, COUNTX### Condicionales- IF, SWITCH, IFERROR, AND, OR, NOT### Texto- CONCATENATE, LEFT, RIGHT, MID, UPPER, LOWER, FORMAT, SUBSTITUTE### Inteligencia de tiempo- TOTALYTD, SAMEPERIODLASTYEAR, DATEADD, DATESMTD, DATESQTD, DATESYTD### Estadísticas- STDEV.P, STDEV.S, VAR.P, VAR.S### Utilidad- DIVIDE, FILTER, SUMMARIZE, DISTINCT, VALUES